# 02_preprocessing
This notebook loads the merged dataset and performs data cleaning/preprocessing.

In [2]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
import warnings
warnings.filterwarnings('ignore')

print("✅ Imports done!")

✅ Imports done!


#  Load Merged Data

In [3]:
df = pd.read_csv('../data/processed/emails_raw_merged.csv')
print(f"Loaded {len(df):,} emails")
print(df['label'].value_counts())

Loaded 10,706 emails
label
1    6615
0    4091
Name: count, dtype: int64


# Handle Missing Values

In [4]:
print(f"Rows before : {len(df):,}")

# Drop rows where body is missing — body is our main feature
df = df.dropna(subset=['body'])

# Fill missing subject with empty string
df['subject'] = df['subject'].fillna('')

df = df.reset_index(drop=True)
print(f"Rows after  : {len(df):,}")

Rows before : 10,706
Rows after  : 10,705


# Remove Duplicates

In [5]:
print(f"Before dedup: {len(df):,}")
df = df.drop_duplicates(subset=['body'], keep='first')
df = df.reset_index(drop=True)
print(f"After dedup : {len(df):,}")

Before dedup: 10,705
After dedup : 10,703


#  Combine Subject + Body

In [6]:
# Subject often has strong phishing signals like "URGENT", "YOU WON"
df['full_text'] = df['subject'].astype(str) + ' ' + df['body'].astype(str)
print("✅ Created full_text = subject + body")

✅ Created full_text = subject + body


 # Text Cleaning Function

In [7]:
def clean_text(text):
    text = str(text).lower()                          # lowercase
    text = re.sub(r'<[^>]+>', ' ', text)              # remove HTML tags
    text = re.sub(r'http\S+|www\.\S+', ' ', text)     # remove URLs
    text = re.sub(r'\S+@\S+', ' ', text)              # remove email addresses
    text = re.sub(r'\d+', ' ', text)                  # remove numbers
    text = re.sub(r'[^\w\s]', ' ', text)              # remove punctuation
    text = re.sub(r'_', ' ', text)                    # remove underscores
    text = re.sub(r'\s+', ' ', text).strip()          # remove extra spaces
    return text

# Quick test
sample = "URGENT! Click http://fake.com to verify your account! Email: fraud@scam.com"
print("Before:", sample)
print("After :", clean_text(sample))

Before: URGENT! Click http://fake.com to verify your account! Email: fraud@scam.com
After : urgent click to verify your account email


# Apply Cleaning

In [8]:
print("Cleaning text... please wait (30-60 seconds)")
df['cleaned_text'] = df['full_text'].apply(clean_text)
print("✅ Done!")

Cleaning text... please wait (30-60 seconds)
✅ Done!


# NLP Preprocessing

In [9]:
lemmatizer = WordNetLemmatizer()
stop_words  = set(stopwords.words('english'))

# Add email-specific words that don't help detection
stop_words.update({'subject', 'from', 'sent', 'mail', 'email',
                   'http', 'www', 'com', 'would', 'could'})

def nlp_preprocess(text):
    tokens = word_tokenize(str(text))
    tokens = [lemmatizer.lemmatize(w) for w in tokens
              if w not in stop_words and len(w) > 2]
    return ' '.join(tokens)

print("Applying NLP preprocessing... (1-2 minutes)")
df['processed_text'] = df['cleaned_text'].apply(nlp_preprocess)
print("✅ Done!")

Applying NLP preprocessing... (1-2 minutes)
✅ Done!


# Add Numerical Features

In [10]:
df['word_count']        = df['cleaned_text'].apply(lambda x: len(x.split()))
df['char_count']        = df['cleaned_text'].apply(len)
df['unique_word_count'] = df['cleaned_text'].apply(lambda x: len(set(x.split())))
df['has_urls']          = df['urls'].apply(lambda x: 1 if x > 0 else 0)
df['subject_length']    = df['subject'].apply(lambda x: len(str(x).split()))

print("Numerical features added:")
print(df[['word_count','char_count','unique_word_count','has_urls','subject_length']].describe().round(1))

Numerical features added:
       word_count  char_count  unique_word_count  has_urls  subject_length
count     10703.0     10703.0            10703.0   10703.0         10703.0
mean        403.8      2316.4              189.3       0.7             5.0
std        6312.0     39609.8             2486.4       0.5             5.4
min           0.0         0.0                0.0       0.0             0.0
25%         103.0       572.0               69.0       0.0             3.0
50%         214.0      1185.0              126.0       1.0             4.0
75%         432.0      2378.5              216.0       1.0             7.0
max      648796.0   4070633.0           254844.0       1.0           483.0


# Save Clean Dataset

In [11]:
# Remove rows where processed text is too short to be useful
df = df[df['processed_text'].str.strip().str.len() > 10].reset_index(drop=True)

# Keep only what we need
cols = ['label','source','processed_text','word_count',
        'char_count','unique_word_count','has_urls','subject_length']
df[cols].to_csv('../data/processed/emails_cleaned.csv', index=False)

print(f"✅ Saved → data/processed/emails_cleaned.csv")
print(f"Final rows : {len(df):,}")
print(f"Phishing   : {(df['label']==1).sum():,}")
print(f"Legitimate : {(df['label']==0).sum():,}")
print()
print("🎉 Preprocessing complete! Move to Notebook 03.")

✅ Saved → data/processed/emails_cleaned.csv
Final rows : 10,700
Phishing   : 6,609
Legitimate : 4,091

🎉 Preprocessing complete! Move to Notebook 03.
